# 01 · Preparación de datos

Datos mensuales del MAGA · Mercado La Terminal · Enero 1998–julio 2026.

## 1. Configuración y lectura

In [7]:
import os
from pathlib import Path
from zipfile import ZipFile
import numpy as np
import pandas as pd

# Definimos dónde están los archivos del proyecto.
try:
    from google.colab import drive
    drive.mount('/content/drive')
    RAIZ = Path('/content/drive/MyDrive/MAGA_Analisis_Fase2')
except ImportError:
    RAIZ = Path(os.environ.get('MAGA_PROJECT_DIR', Path.cwd() / 'MAGA_Analisis_Fase2'))

ORIGEN = RAIZ / 'data' / 'original'
DESTINO = RAIZ / 'data' / 'prepared' / 'maga_precios_mensuales_la_terminal_1998_2026.csv'
DESTINO.parent.mkdir(parents=True, exist_ok=True)

# Buscamos el ZIP mensual y cargamos su archivo CSV.
archivos = sorted(p for p in ORIGEN.glob('*.zip') if 'mensual' in p.name.casefold() and 'csv' in p.name.casefold())
if not archivos:
    raise FileNotFoundError(f'Coloca el ZIP mensual CSV del MAGA en {ORIGEN}')
zip_origen = next((p for p in archivos if 'octubre 2026' in p.name.casefold()), archivos[-1])
with ZipFile(zip_origen) as z:
    csvs = [n for n in z.namelist() if n.lower().endswith('.csv')]
    if len(csvs) != 1:
        raise ValueError('El ZIP debe contener un único CSV de precios mensuales.')
    with z.open(csvs[0]) as f:
        fuente = pd.read_csv(f, encoding='utf-8-sig', low_memory=False)

# Confirmamos que el archivo contiene los campos que necesitamos.
columnas = ['Actor', 'Periodicidad', 'Mercado', 'Producto', 'Medida', 'Fecha', 'Moneda', 'Precio']
if not set(columnas).issubset(fuente.columns):
    raise ValueError('La estructura del archivo MAGA no coincide con la esperada.')
print(f'Archivo: {zip_origen.name} | Registros: {len(fuente):,}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Archivo: Precios mensuales de diversos productos agrícolas en Guatemala a octubre 2026 CSV.zip | Registros: 45,622


## 2. Selección y validación de las series

In [8]:
# Estos son los tres productos y sus unidades de venta.
productos = {
    'Tomate de cocina, grande, de primera': 'Caja (45 - 50 lb)',
    'Papa Loman, lavada, grande, de primera': 'Quintal',
    'Chile Pimiento, grande, de primera': 'Caja (90 -100 unidades)',
}

# Seleccionamos solo los registros que corresponden a nuestro estudio.
filtro = (
    fuente['Actor'].eq('Mayorista') & fuente['Periodicidad'].eq('Mensual')
    & fuente['Mercado'].eq('La Terminal') & fuente['Moneda'].eq('GTQ')
    & fuente['Producto'].isin(productos)
    & fuente['Medida'].eq(fuente['Producto'].map(productos))
)
seleccion = fuente.loc[filtro, columnas].copy()
# Convertimos fechas y precios al formato adecuado para analizarlos.
seleccion['Fecha'] = pd.to_datetime(seleccion['Fecha'], format='%Y-%m-%d', errors='coerce')
seleccion['Precio'] = pd.to_numeric(seleccion['Precio'], errors='coerce')
seleccion = seleccion.loc[seleccion['Fecha'].between('1998-01-01', '2026-07-01')].copy()

# Verificamos valores inválidos, duplicados y continuidad mensual.
if seleccion['Fecha'].isna().any() or seleccion['Precio'].isna().any():
    raise ValueError('Hay fechas o precios faltantes/incorrectos.')
if not np.isfinite(seleccion['Precio']).all() or (seleccion['Precio'] <= 0).any():
    raise ValueError('Los precios deben ser positivos y finitos.')
if seleccion.duplicated(['Producto', 'Fecha']).any():
    raise ValueError('Se detectaron registros mensuales duplicados.')

calendario = pd.date_range('1998-01-01', '2026-07-01', freq='MS')
for producto in productos:
    fechas = pd.DatetimeIndex(seleccion.loc[seleccion['Producto'].eq(producto), 'Fecha'].sort_values())
    if not fechas.equals(calendario):
        raise ValueError(f'Serie mensual incompleta o desordenada: {producto}')

resumen = (seleccion.groupby(['Producto', 'Medida'])
           .agg(Meses=('Fecha', 'size'), Inicio=('Fecha', 'min'), Fin=('Fecha', 'max'))
           .reset_index())
display(resumen)

,Producto,Medida,Meses,Inicio,Fin
0,"Chile Pimiento, grande, de primera",Caja (90 -100 unidades),343,1998-01-01,2026-07-01
1,"Papa Loman, lavada, grande, de primera",Quintal,343,1998-01-01,2026-07-01
2,"Tomate de cocina, grande, de primera",Caja (45 - 50 lb),343,1998-01-01,2026-07-01


## 3. Preparación y exportación

In [9]:
# Ordenamos las series y añadimos el año, mes y período.
preparado = seleccion.sort_values(['Producto', 'Fecha']).reset_index(drop=True)
preparado['Anio'] = preparado['Fecha'].dt.year
preparado['Mes'] = preparado['Fecha'].dt.month
preparado['Periodo'] = preparado['Fecha'].dt.strftime('%Y-%m')
# Guardamos el archivo que utilizarán los otros dos notebooks.
preparado.to_csv(DESTINO, index=False, encoding='utf-8-sig', date_format='%Y-%m-%d')

# Comprobación mínima del archivo consumido por los Notebooks 02 y 03.
lectura = pd.read_csv(DESTINO, parse_dates=['Fecha'], encoding='utf-8-sig')
assert len(lectura) == 1029 and lectura['Producto'].nunique() == 3
assert lectura['Precio'].gt(0).all() and lectura['Fecha'].max() == pd.Timestamp('2026-07-01')
print(f'Archivo preparado: {DESTINO} | {len(lectura):,} registros')
display(lectura.head(3))

Archivo preparado: /content/drive/MyDrive/MAGA_Analisis_Fase2/data/prepared/maga_precios_mensuales_la_terminal_1998_2026.csv | 1,029 registros


,Actor,Periodicidad,Mercado,Producto,Medida,Fecha,Moneda,Precio,Anio,Mes,Periodo
0,Mayorista,Mensual,La Terminal,"Chile Pimiento, grande, de primera",Caja (90 -100 unidades),1998-01-01,GTQ,70.00,1998,1,1998-01
1,Mayorista,Mensual,La Terminal,"Chile Pimiento, grande, de primera",Caja (90 -100 unidades),1998-02-01,GTQ,43.75,1998,2,1998-02
2,Mayorista,Mensual,La Terminal,"Chile Pimiento, grande, de primera",Caja (90 -100 unidades),1998-03-01,GTQ,44.00,1998,3,1998-03
